In [1]:
import pandas as pd
import datetime as datetime

In [2]:
def to_time(str_time):
	if (len(str_time) < 10) : return str_time
	else : return str_time[-9:]

In [3]:
def to_date(str_date):
	return str_date[:10]

In [4]:
def merge_time_date(str_time):
	if len(str_time.split(".")) > 1 :
		return datetime.datetime.strptime(str_time, '%Y-%m-%d %H:%M:%S.%f')
	else:
		return datetime.datetime.strptime(str_time, '%Y-%m-%d %H:%M:%S')

In [5]:
peak_file = "../sound_proj/data/Processed-data/output.csv"

flights = pd.read_excel("../sound_proj/data/Raw-data/full_flights.xlsx", dtype={'Local Date': str, 'Local Time': str})
peaks = pd.read_csv(peak_file)

print ("Reading Flight File Successfully.")

Reading Flight File Successfully.


In [6]:
flights["Local Time"] = flights["Local Time"].apply(lambda a: to_time(a))
flights["Local Date"] = flights["Local Date"].apply(lambda a: to_date(a))

flights["Datetime"] = flights["Local Date"] + " " + flights["Local Time"]
flights["Datetime"] = flights["Datetime"].apply(lambda a: merge_time_date(a))

peaks["start_time"] = peaks["start_time"].apply(lambda a: merge_time_date(a))
peaks["end_time"]   = peaks["end_time"].apply(lambda a: merge_time_date(a))

In [7]:
flights.head()

,CALLSIGN,REGISTRATION,ACTYPE,aircraftTypeinINM,APPDEPFLAG,POD,DEST,ATA,ATA2,Local Time,Local Date,VIA,Unnamed: 12,RWY,WTC,Datetime
0,MAS797,9MMXB,B738,737800,D,VTBS,WMKK,2013.12.30,23:31:00,06:31:00,2013-12-31,REG01LD,3,01L,M,2013-12-31 06:31:00
1,THA8629,HSTKC,B773,777300,A,UUDD,VTBS,2013.12.30,23:32:00,06:32:00,2013-12-31,LIM01RA,2,01R,H,2013-12-31 06:32:00
2,GIA865,PKGMF,B738,737800,D,VTBS,WIII,2013.12.30,23:35:00,06:35:00,2013-12-31,RYN01LD,2,01L,M,2013-12-31 06:35:00
3,THA338,HSTEA,A333,A330-343,A,VOMM,VTBS,2013.12.30,23:36:00,06:36:00,2013-12-31,NaN,2,01R,H,2013-12-31 06:36:00
4,BKP1101,HSPGT,A319,A319-131,D,VTBS,VTSM,2013.12.30,23:39:00,06:39:00,2013-12-31,REG01RD,3,01R,M,2013-12-31 06:39:00


In [8]:
peaks.head()

,start_time,end_time,peak_time,interval
0,2014-03-12 19:53:58.700,2014-03-12 19:54:23.100,41089,24.4
1,2014-03-12 20:41:14.200,2014-03-12 20:41:35.200,69427,21.0
2,2014-03-12 20:45:33.400,2014-03-12 20:45:59.800,72046,26.4
3,2014-03-12 21:20:27.900,2014-03-12 21:20:47.600,92957,19.7
4,2014-03-12 21:30:25.900,2014-03-12 21:30:44.700,98933,18.8


In [9]:
flights = flights.drop(['REGISTRATION', 'ATA', 'ATA2', 'Unnamed: 12'], axis=1)

In [10]:
html = pd.read_html('https://en.wikipedia.org/wiki/List_of_airline_codes')
df_CALLSIGN_NAME = html[0]
df_CALLSIGN_NAME = df_CALLSIGN_NAME.dropna(subset=['ICAO'])
df_CALLSIGN_NAME.head()

,IATA,ICAO,Airline,Call sign,Country/Region,Comments
0,PR,BOI,2GO,ABAIR,Philippines,NaN
1,NaN,EVY,"34 Squadron, Royal Australian Air Force",Multiple,NaN,NaN
2,NaN,GNL,135 Airways,GENERAL,United States,NaN
16,2T,TBS,Timbis Air,TIMBIS,Kenya,NaN
17,3F,FIE,FlyOne Armenia,ARMRIDER,Armenia,NaN


In [11]:
airline_names = dict(zip(df_CALLSIGN_NAME['ICAO'], df_CALLSIGN_NAME['Airline']))

In [12]:
def callsign_name(call):
    c_name = ''.join(i for i in call if not i.isdigit())
    if c_name in list(df_CALLSIGN_NAME['ICAO']):
        return airline_names[c_name]
    else:
        return c_name

In [13]:
df_airport_names = pd.read_csv('../sound_proj/data/aircraft-data/iata-icao.csv')
df_airport_names = df_airport_names.dropna(subset=['icao'])
df_airport_names.head()

,country_code,region_name,iata,icao,airport,latitude,longitude
0,AE,Ash Shariqah,SHJ,OMSJ,Sharjah International Airport,25.3286,55.5172
1,AE,Abu Zaby,AZI,OMAD,Al Bateen Executive Airport,24.4283,54.4581
2,AE,Al Fujayrah,FJR,OMFJ,Fujairah International Airport,25.1122,56.3240
3,AE,Abu Zaby,XSB,OMBY,Sir Bani Yas Airport,24.2836,52.5803
4,AE,Ra's al Khaymah,RKT,OMRK,Ras Al Khaimah International Airport,25.6135,55.9388


In [14]:
airport_names = dict(zip(df_airport_names['icao'], df_airport_names['airport']))

In [15]:
def call_airport_name(name):
    if name in list(df_airport_names['icao']):
        return airport_names[name]
    else:
        return name

In [16]:
def phasedecide(date):
    dt = date.time()
    if (dt.hour >= 7) & (dt.hour < 19):
        return 1
    elif (dt.hour >= 19) & (dt.hour < 22):
        return 2
    else:
        return 3

In [17]:
min_peak, max_peak = min(peaks["start_time"]), max(peaks["end_time"])
flights = flights[(flights['Datetime'] > min_peak) & (flights['Datetime'] < max_peak)]
flights.head()

,CALLSIGN,ACTYPE,aircraftTypeinINM,APPDEPFLAG,POD,DEST,Local Time,Local Date,VIA,RWY,WTC,Datetime
61713,TNA867,A333,A330-343,A,RCTP,VTBS,19:55:00,2014-03-12,NaN,19R,H,2014-03-12 19:55:00
61714,CSZ9024,B738,737800,D,VTBS,ZGSZ,19:57:00,2014-03-12,SEL19RD,19R,M,2014-03-12 19:57:00
61715,RBA517,A320,A320-211,A,WBSB,VTBS,19:57:00,2014-03-12,GOM19LA,19L,M,2014-03-12 19:57:00
61716,UAE418,B77W,777300,D,VTBS,YSSY,19:58:00,2014-03-12,BUT19RD,19R,H,2014-03-12 19:58:00
61717,AMU882,A321,A321-232,A,VMMC,VTBS,20:00:00,2014-03-12,NaN,19L,M,2014-03-12 20:00:00


In [18]:
flights['phase'] = flights['Datetime'].apply(lambda a: phasedecide(a))
flights["airline_name"] = flights['CALLSIGN'].apply(lambda a: callsign_name(a))

flights['POD_NAME'] = flights['POD'].apply(lambda a: call_airport_name(a))
flights['DEST_NAME'] = flights['DEST'].apply(lambda a: call_airport_name(a))

flights['POD_to_DEST'] = flights['POD_NAME'] + "➔" + flights['DEST_NAME']

In [49]:
lag_width = 30 # seconds

In [50]:
def match_cond(start,stop,date_time):
    return (date_time >= start-datetime.timedelta(milliseconds=lag_width*1000)) & (date_time <= stop+datetime.timedelta(milliseconds=lag_width*1000))

In [53]:
matched = [flights[match_cond(start,stop,flights['Datetime'])] for start, stop in zip(peaks['start_time'], peaks['end_time'])
           if len(flights[match_cond(start,stop,flights['Datetime'])]) > 0]

In [46]:
len(matched) / len(peaks)

0.9851851851851852

In [23]:
df_matched = pd.concat(matched).reset_index(drop=True)
df_matched.head()

,CALLSIGN,ACTYPE,aircraftTypeinINM,APPDEPFLAG,POD,DEST,Local Time,Local Date,VIA,RWY,WTC,Datetime,phase,airline_name,POD_NAME,DEST_NAME,POD_to_DEST
0,THA2011,A320,A320-211,A,VTUD,VTBS,20:41:00,2014-03-12,NaN,19L,M,2014-03-12 20:41:00,2,Thai Airways International,Udon Thani International Airport,Suvarnabhumi Airport,Udon Thani International Airport➔Suvarnabhumi ...
1,ETD404,B77W,777300,A,OMAA,VTBS,20:42:00,2014-03-12,BET19RA,19R,H,2014-03-12 20:42:00,2,Etihad Airways,Abu Dhabi International Airport,Suvarnabhumi Airport,Abu Dhabi International Airport➔Suvarnabhumi A...
2,BCC867,B763,767300,A,RKSI,VTBS,20:46:00,2014-03-12,NaN,19R,H,2014-03-12 20:46:00,2,BCC,Incheon International Airport,Suvarnabhumi Airport,Incheon International Airport➔Suvarnabhumi Air...
3,THA665,A333,A330-343,A,ZSPD,VTBS,21:21:00,2014-03-12,NaN,19R,H,2014-03-12 21:21:00,2,Thai Airways International,Shanghai Pudong International Airport,Suvarnabhumi Airport,Shanghai Pudong International Airport➔Suvarnab...
4,JNA001,B738,737800,A,RKSI,VTBS,21:31:00,2014-03-12,NaN,19R,M,2014-03-12 21:31:00,2,Jin Air,Incheon International Airport,Suvarnabhumi Airport,Incheon International Airport➔Suvarnabhumi Air...


In [24]:
df_flight = pd.read_csv('../sound_proj/data/Processed-data/number_aircrafts.csv')
df_flight.head()

,start_time,end_time,aircrafts
0,2014-03-12 18:45:42,2014-03-12 19:45:42,3
1,2014-03-12 19:45:42,2014-03-12 20:45:42,3
2,2014-03-12 20:45:42,2014-03-12 21:45:42,2
3,2014-03-12 21:45:42,2014-03-12 22:45:42,5
4,2014-03-12 22:45:42,2014-03-12 23:45:42,7


In [25]:
df_flight['start_time'], df_flight['end_time'] = pd.to_datetime(df_flight['start_time']), pd.to_datetime(df_flight['end_time'])

In [26]:
n_aircraft_pred = [len(df_matched[(df_matched['Datetime'] >= start) & (df_matched['Datetime'] <= stop)]) for start, stop in zip(df_flight['start_time'], df_flight['end_time'])]

In [27]:
df_flight['aircrafts_pred'] = n_aircraft_pred

In [28]:
df_flight.head()

,start_time,end_time,aircrafts,aircrafts_pred
0,2014-03-12 18:45:42,2014-03-12 19:45:42,3,0
1,2014-03-12 19:45:42,2014-03-12 20:45:42,3,2
2,2014-03-12 20:45:42,2014-03-12 21:45:42,2,3
3,2014-03-12 21:45:42,2014-03-12 22:45:42,5,5
4,2014-03-12 22:45:42,2014-03-12 23:45:42,7,8


In [29]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(df_flight['aircrafts'], df_flight['aircrafts_pred'])
print(mse)

235.2202380952381
